In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna


In [ ]:
def objective(trial, X_train, y_train, X_test, y_test):
    """Функция для оптимизации гиперпараметров"""
    
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    
    return mse

In [51]:
def optimize_hyperparameters(X_train, y_train, X_test, y_test, n_trials=100):
    """Оптимизация гиперпараметров с помощью Optuna"""
    
    print(f"\nЗапуск оптимизации гиперпараметров ({n_trials} trials)...")
    
    study = optuna.create_study(direction='minimize')
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_test, y_test), 
        n_trials=n_trials
    )
    
    print("\nЛучшие гиперпараметры:")
    for key, value in study.best_params.items():
        print(f"{key}: {value}")
    print(f"Лучшее MSE: {study.best_value:.4f}")
    
    return study.best_params

In [52]:
def train_final_model(X_train, y_train, best_params):
    """Обучение финальной модели с лучшими параметрами"""
    
    model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    return model

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Оценка качества модели"""
    
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    print("\n" + "="*50)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ")
    print("="*50)
    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"R² Score: {r2:.4f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mae:.2f}%")
    
    return y_pred, {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape}

In [ ]:
def plot_residuals(y_test, y_pred):
    """Построение графиков остатков"""
    
    residuals = y_test - y_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    
    # График 1: Predicted vs Actual
    axes[0].scatter(y_pred, y_test, alpha=0.6, color='blue')
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0].set_xlabel('Predicted Values')
    axes[0].set_ylabel('Actual Values')
    axes[0].set_title('Predicted vs Actual Values')
    axes[0].grid(True, alpha=0.3)
    
    # График 2: Residuals vs Predicted
    axes[1].scatter(y_pred, residuals, alpha=0.6, color='green')
    axes[1].axhline(y=0, color='r', linestyle='--')
    axes[1].set_xlabel('Predicted Values')
    axes[1].set_ylabel('Residuals')
    axes[1].set_title('Residuals vs Predicted Values')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    return residuals

In [58]:
dataset = fetch_california_housing(as_frame=True)

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

best_params = optimize_hyperparameters(X_train, y_train, X_test, y_test, n_trials=50)

final_model = train_final_model(X_train, y_train, best_params)

y_pred, metrics = evaluate_model(final_model, X_test, y_test)

[I 2025-10-07 13:22:46,573] A new study created in memory with name: no-name-2eae0514-f382-4a75-89aa-19ade2015a12



Запуск оптимизации гиперпараметров (50 trials)...


[I 2025-10-07 13:22:47,142] Trial 0 finished with value: 0.49982125309214004 and parameters: {'n_estimators': 213, 'max_depth': 5, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.49982125309214004.
[I 2025-10-07 13:22:47,952] Trial 1 finished with value: 0.272593210135899 and parameters: {'n_estimators': 148, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.272593210135899.
[I 2025-10-07 13:22:52,426] Trial 2 finished with value: 0.27158592219545896 and parameters: {'n_estimators': 279, 'max_depth': 18, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 2 with value: 0.27158592219545896.
[I 2025-10-07 13:22:52,739] Trial 3 finished with value: 0.6247390651535494 and parameters: {'n_estimators': 113, 'max_depth': 3, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 0.2715859221954


Лучшие гиперпараметры:
n_estimators: 108
max_depth: 19
min_samples_split: 2
min_samples_leaf: 2
max_features: log2
Лучшее MSE: 0.2425

РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ
Mean Squared Error (MSE): 0.2425
Root Mean Squared Error (RMSE): 0.4925
Mean Absolute Error (MAE): 0.3215
R² Score: 0.8149
Mean Absolute Percentage Error (MAPE): 0.32%
